# CARE++ Evidence Layer v2 — Adaptive Retrieval, Multi-Factor Ranking, Conflict-Aware Fusion

This is now self-contained — Section 0 extracts, chunks, and embeds your PDFs
itself (or loads the saved results if you've already run it once), rather
than depending on a separate `CARE___evidence_layer.ipynb`. It also needs,
from `CARE_FL_Version_2.ipynb`:
- `care_fl_predictions_test.parquet` (the test-set predictions from your best
  federated model)

It replaces the retrieval → ranking → fusion → LLM-input section (roughly
your cells 18–45) with a version closer to what your project write-up
describes: retrieval strategy that actually changes with the situation,
ranking with named/traceable factors instead of two fixed weights, evidence
fusion that flags disagreement instead of silently averaging it away, and a
lightweight claim-extraction step so the LLM sees claims rather than whole
1200-character chunks.


## 0. Extract PDFs and build the evidence corpus (or load it, if already done)

Upload your 6 PDFs to `/content/` first, then run this cell. It checks
whether `care_evidence_chunks.parquet`/`care_evidence_embeddings.npy` already
exist — if so, it just loads them (fast path for reruns). If not, it extracts
text page-by-page from each PDF (handling encryption automatically),
chunks long pages with overlap, embeds every chunk, and saves both files
for next time.

**If you add or change a PDF, set `FORCE_REEXTRACT = True`** — otherwise
this will silently keep using the old cached chunks/embeddings and never
notice your corpus changed.

In [1]:
pip install pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 385.1/385.1 kB 12.1 MB/s eta 0:00:00


In [2]:
!pip install -U "transformers==4.57.3" "accelerate>=1.2.0" "bitsandbytes>=0.50.0"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 107.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 18.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 42.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 106.2 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.28.0
    Uninstalling huggingface_hub-1.28.0:
      Successfully uninstalled huggingface_hub-1.28.0
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.23.1
    Uninstalling tokenizers-0.23.1:
      Successfully uninstalled tokenizers-0.23.1
  Attempting uninstall: transformers
    Found existing installation: transformers 5.16.1
    Uninstalling transformers-5.16.1:
      Successfully uninstalled transformers-5.16.1
ERROR: pip's dependency resolver does not currently take into

In [3]:
import os
import re
import json
import numpy as np
import pandas as pd
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer

EVIDENCE_PATH = "/content/care_evidence_chunks.parquet"
EMBEDDING_PATH = "/content/care_evidence_embeddings.npy"
PREDICTION_PATH = "/content/care_fl_predictions_test.parquet"

FORCE_REEXTRACT = False  # set True after adding/changing a PDF

CHUNK_SIZE = 1200
CHUNK_OVERLAP = 200

# Labels verified earlier to route correctly through classify_source().
# Adjust the right-hand paths to match whatever you actually uploaded.
PDF_FILES = {
    "ICRP_Publication_103":         "/content/ICRP_Publication_103-Annals_of_the_ICRP_37(2-4)-Free_extract.pdf",
    "IAEA":                          "/content/IAEA.pdf",
    "EHS_Stanford_Safety_Manual":    "/content/RadiationSafetyManualrev04.16.25.pdf",
    "Secondary_Review_Yadeta2024":   "/content/radiology-12-1106.pdf",
    "IAEA_Case_Istanbul":            "/content/Pub1102_web.pdf",
    "NRC_10CFR20":                   "/content/10 CFR Part 20 (up to date as of 9-01-2026).pdf",
}


def chunk_page_text(text, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    """Sliding-window chunking WITHIN a single page, so every chunk's page
    number is exact -- never split across pages, which would make citation
    validation ambiguous about which page a claim actually came from."""
    text = text.strip()
    if len(text) <= chunk_size:
        return [text] if text else []
    chunks = []
    start = 0
    while start < len(text):
        chunks.append(text[start:start + chunk_size])
        start += chunk_size - overlap
    return chunks


def extract_pdf(label, path):
    """Extract page-level chunks from one PDF. Returns a list of records
    and never raises -- a failure on one file shouldn't kill the whole
    corpus build, but it IS printed loudly so you notice."""
    records = []
    try:
        reader = PdfReader(path)
        if reader.is_encrypted:
            result = reader.decrypt("")
            if result == 0:
                print(f"  FAILED: {label} is encrypted and empty-password decrypt failed -- "
                      "this file needs its actual password, skipping.")
                return records

        pages_with_text = 0
        for page_num, page in enumerate(reader.pages, start=1):
            text = page.extract_text() or ""
            if not text.strip():
                continue
            pages_with_text += 1
            for chunk_text in chunk_page_text(text):
                records.append({"source": label, "page": page_num, "text": chunk_text})

        print(f"  OK: {label} -- {len(reader.pages)} pages, {pages_with_text} with extractable "
              f"text, {len(records)} chunks")
        if pages_with_text < len(reader.pages) * 0.5:
            print(f"    WARNING: less than half the pages yielded text -- possibly scanned/image-based "
                  f"pages without OCR, worth checking manually.")

    except Exception as e:
        print(f"  FAILED: {label} -- {type(e).__name__}: {e}")

    return records


need_extraction = FORCE_REEXTRACT or not (os.path.exists(EVIDENCE_PATH) and os.path.exists(EMBEDDING_PATH))

if not need_extraction:
    print("Found existing evidence_chunks/evidence_embeddings -- loading (fast path).")
    print("(Set FORCE_REEXTRACT = True above if you've added/changed a PDF.)")
    evidence_chunks = pd.read_parquet(EVIDENCE_PATH)
    evidence_embeddings = np.load(EMBEDDING_PATH)
else:
    print("=" * 70)
    print("EXTRACTING PDFs")
    print("=" * 70)

    all_records = []
    for label, path in PDF_FILES.items():
        if not os.path.exists(path):
            print(f"  MISSING FILE: {label} -> {path} not found, skipping.")
            continue
        all_records.extend(extract_pdf(label, path))

    if not all_records:
        raise RuntimeError("No chunks extracted from any PDF -- check PDF_FILES paths above.")

    evidence_chunks = pd.DataFrame(all_records)
    print(f"\nTotal chunks across all sources: {len(evidence_chunks)}")
    print(evidence_chunks["source"].value_counts())

    print("\n" + "=" * 70)
    print("EMBEDDING CHUNKS")
    print("=" * 70)
    embedding_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
    evidence_embeddings = embedding_model.encode(
        evidence_chunks["text"].tolist(),
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=True,
    )

    evidence_chunks.to_parquet(EVIDENCE_PATH, index=False)
    np.save(EMBEDDING_PATH, evidence_embeddings)
    print(f"\nSaved {EVIDENCE_PATH} and {EMBEDDING_PATH}")

if "embedding_model" not in globals():
    embedding_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

prediction_results = pd.read_parquet(PREDICTION_PATH)

print("\nEvidence chunks:", evidence_chunks.shape)
print("Evidence embeddings:", evidence_embeddings.shape)
print("Prediction rows:", prediction_results.shape)


EXTRACTING PDFs
  OK: ICRP_Publication_103 -- 35 pages, 35 with extractable text, 97 chunks
  OK: IAEA -- 471 pages, 466 with extractable text, 1076 chunks
  OK: EHS_Stanford_Safety_Manual -- 105 pages, 104 with extractable text, 259 chunks
  OK: Secondary_Review_Yadeta2024 -- 2 pages, 2 with extractable text, 11 chunks
  OK: IAEA_Case_Istanbul -- 86 pages, 85 with extractable text, 168 chunks
  OK: NRC_10CFR20 -- 158 pages, 158 with extractable text, 354 chunks

Total chunks across all sources: 1965
source
IAEA                           1076
NRC_10CFR20                     354
EHS_Stanford_Safety_Manual      259
IAEA_Case_Istanbul              168
ICRP_Publication_103             97
Secondary_Review_Yadeta2024      11
Name: count, dtype: int64

EMBEDDING CHUNKS


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/62 [00:00<?, ?it/s]


Saved /content/care_evidence_chunks.parquet and /content/care_evidence_embeddings.npy

Evidence chunks: (1965, 3)
Evidence embeddings: (1965, 384)
Prediction rows: (355856, 14)


## 1. Context Builder (extends your existing `context_df` construction)

In [4]:
def build_context_df(prediction_results):
    """Same context-enrichment logic as your original notebook, kept in one
    place so this notebook is self-contained."""
    context_df = prediction_results.copy()
    context_df["timestamp"] = pd.to_datetime(context_df["timestamp"])

    context_df["prediction_residual"] = context_df["current_cpm"] - context_df["predicted_cpm"]
    context_df["residual_abs"] = context_df["prediction_residual"].abs()
    context_df["relative_prediction_error"] = (
        context_df["residual_abs"] / context_df["current_cpm"].replace(0, np.nan)
    ).fillna(0.0)

    context_df["prediction_error_direction"] = np.where(
        context_df["prediction_residual"] > 0, "underprediction",
        np.where(context_df["prediction_residual"] < 0, "overprediction", "accurate"),
    )

    context_df["hour"] = context_df["timestamp"].dt.hour
    context_df["day_of_week"] = context_df["timestamp"].dt.dayofweek
    context_df["day_name"] = context_df["timestamp"].dt.day_name()
    context_df["is_weekend"] = (context_df["day_of_week"] >= 5).astype(int)

    def classify_time_period(hour):
        if 5 <= hour < 12:
            return "morning"
        elif 12 <= hour < 17:
            return "afternoon"
        elif 17 <= hour < 22:
            return "evening"
        return "night"

    context_df["time_period"] = context_df["hour"].apply(classify_time_period)

    # "trend" existed in your original pipeline (built earlier from cpm_change).
    # If it isn't present here, derive a simple version from the residual sign
    # so this notebook can run standalone.
    if "trend" not in context_df.columns:
        context_df = context_df.sort_values(["device_id", "timestamp"]).reset_index(drop=True)
        context_df["previous_cpm"] = context_df.groupby("device_id")["current_cpm"].shift(1)
        cpm_change = context_df["current_cpm"] - context_df["previous_cpm"]
        context_df["trend"] = np.select(
            [cpm_change > 0, cpm_change < 0], ["increasing", "decreasing"], default="stable",
        )

    return context_df


context_df = build_context_df(prediction_results)
print("Context rows:", context_df.shape)


Context rows: (355856, 20)


## 2. Severity tiering (drives the *adaptive* part of retrieval)

`current_cpm` bands below are **placeholders**, not calibrated safety
thresholds — see the safeguard note above. `relative_prediction_error` can
escalate the tier by one level: a measurement the model badly failed to
predict gets treated as needing broader, more cautious evidence retrieval,
independent of whether the raw CPM value itself looks large.

In [5]:
# EXTENSION POINT: replace these bands with cited IAEA/ICRP reference levels
# before using severity tiers for anything beyond retrieval routing.
SEVERITY_CPM_BANDS = [
    (0.0, 1.0, "low"),
    (1.0, 5.0, "moderate"),
    (5.0, 20.0, "high"),
    (20.0, float("inf"), "critical"),
]

SEVERITY_ORDER = ["low", "moderate", "high", "critical"]

RELATIVE_ERROR_ESCALATION_THRESHOLD = 0.5  # a >50% miss escalates the tier


def classify_severity(row):
    """Routing-only tier. NEVER present this to an end user as a safety verdict."""
    cpm = float(row["current_cpm"])

    base_tier = "critical"
    for low, high, tier in SEVERITY_CPM_BANDS:
        if low <= cpm < high:
            base_tier = tier
            break

    if row.get("relative_prediction_error", 0.0) > RELATIVE_ERROR_ESCALATION_THRESHOLD:
        idx = min(SEVERITY_ORDER.index(base_tier) + 1, len(SEVERITY_ORDER) - 1)
        return SEVERITY_ORDER[idx]

    return base_tier


context_df["severity_tier"] = context_df.apply(classify_severity, axis=1)
print(context_df["severity_tier"].value_counts())


severity_tier
critical    224449
high        131407
Name: count, dtype: int64


## 3. Context Builder: structured situation profile + query text

In [6]:
def build_situation_profile(row):
    """Structured context profile (the analogue of your project notes'
    Occupation / Location / Exposure Duration table).

    - `lat`/`lon`: populated from Section 1b's GPS join in the FL notebook,
      when present. Real, measured data — not simulated.
    - `occupation`, `radiation_type`: no sensor for these exists; left as
      None deliberately.
    - Physiological context (heart rate, motion, skin temperature): NOT
      included here at all, on purpose. No dataset correlating these with
      real radiation exposure exists (checked — WESAD/PPG-DaLiA have HR/
      motion/temperature but are from unrelated studies with no radiation
      or GPS data and no shared subjects/timestamps to join on). Fabricating
      a join to an unrelated dataset would misrepresent this as measured
      multi-modal fusion. This is Future Work, pending a real wearable
      prototype — see Section 16 Limitations, not a code gap to "extend."
    """
    has_gps = "lat" in row.index and "lon" in row.index and pd.notna(row.get("lat"))

    return {
        "device_id": int(row["device_id"]),
        "timestamp": str(row["timestamp"]),
        "current_cpm": float(row["current_cpm"]),
        "predicted_cpm": float(row["predicted_cpm"]),
        "prediction_residual": float(row["prediction_residual"]),
        "residual_abs": float(row["residual_abs"]),
        "relative_prediction_error": float(row["relative_prediction_error"]),
        "prediction_error_direction": str(row["prediction_error_direction"]),
        "trend": str(row["trend"]),
        "hour": int(row["hour"]),
        "day_of_week": int(row["day_of_week"]),
        "day_name": str(row["day_name"]),
        "is_weekend": int(row["is_weekend"]),
        "time_period": str(row["time_period"]),
        "severity_tier": str(row["severity_tier"]),
        "lat": float(row["lat"]) if has_gps else None,
        "lon": float(row["lon"]) if has_gps else None,
        # EXTENSION POINT: occupation, radiation_type — add once a wearable
        # actually senses them, and thread into build_care_query() below.
        "occupation": None,
        "radiation_type": None,
    }


def build_care_query(profile):
    location_line = (
        f"Location: latitude {profile['lat']:.4f}, longitude {profile['lon']:.4f}.\n"
        if profile.get("lat") is not None else ""
    )

    query = f"""
Radiation exposure assessment and radiation protection guidance.

Situation severity tier (retrieval routing only, not a safety claim): {profile['severity_tier']}

Current radiation measurement: {profile['current_cpm']:.2f} CPM.
Predicted next radiation measurement: {profile['predicted_cpm']:.2f} CPM.
Prediction residual: {profile['prediction_residual']:.2f} CPM.
Absolute prediction error: {profile['residual_abs']:.2f} CPM.
Prediction error direction: {profile['prediction_error_direction']}.
Radiation trend: {profile['trend']}.
Relative prediction error: {profile['relative_prediction_error']:.3f}.
Observation time: {profile['hour']:02d}:00 ({profile['time_period']}).
Day: {profile['day_name']}.
{location_line}
Find authoritative evidence concerning radiation exposure, radiation
protection principles, health effects, exposure assessment, monitoring,
and interpretation that is relevant to this contextual situation.
"""
    return query.strip()


## 4. Adaptive Retrieval

The retrieval strategy (candidate pool size `top_k`, and which keyword
categories get emphasized when re-scoring candidates) changes with the
severity tier, instead of being fixed at `top_k=30` / equal category weights
every time — this is the actual "retrieval strategy changes automatically"
behavior described in your project notes.

In [7]:
CONTEXT_TERMS = {
    "exposure": ["exposure", "exposed", "dose", "radiation"],
    "protection": ["protection", "protective", "optimization", "safety"],
    "monitoring": ["monitoring", "measurement", "monitor", "assessment"],
    "health": ["health", "effect", "tissue", "cancer"],
    "risk": ["risk", "hazard", "detriment"],
    # New: lets retrieval distinguish "give me a past case" and "give me the
    # binding regulation" from general protection guidance.
    "historical": ["accident", "incident", "lessons learned", "case"],
    "regulatory": ["regulation", "requirement", "compliance", "limit", "cfr"],
}

RETRIEVAL_STRATEGY = {
    "low": {
        "top_k": 15,
        "category_weights": {"exposure": 1.0, "protection": 1.0, "monitoring": 1.2, "health": 0.6, "risk": 0.6,
                              "historical": 0.5, "regulatory": 0.8},
    },
    "moderate": {
        "top_k": 20,
        "category_weights": {"exposure": 1.0, "protection": 1.1, "monitoring": 1.0, "health": 0.9, "risk": 0.9,
                              "historical": 0.8, "regulatory": 1.0},
    },
    "high": {
        "top_k": 30,
        "category_weights": {"exposure": 1.0, "protection": 1.3, "monitoring": 0.8, "health": 1.2, "risk": 1.3,
                              "historical": 1.3, "regulatory": 1.1},
    },
    "critical": {
        "top_k": 40,
        "category_weights": {"exposure": 1.0, "protection": 1.5, "monitoring": 0.7, "health": 1.4, "risk": 1.5,
                              "historical": 1.5, "regulatory": 1.2},
    },
}


def contextual_keyword_score(text, category_weights):
    text = text.lower()
    total = 0.0
    for category, terms in CONTEXT_TERMS.items():
        weight = category_weights.get(category, 1.0)
        matches = sum(1 for term in terms if term in text)
        total += weight * matches
    return total


def retrieve_adaptive(profile, evidence_chunks, evidence_embeddings, embedding_model):
    """Severity-aware retrieval: query text, pool size, and keyword
    weighting all depend on the situation profile."""
    strategy = RETRIEVAL_STRATEGY[profile["severity_tier"]]

    query = build_care_query(profile)
    query_embedding = embedding_model.encode(
        [query], convert_to_numpy=True, normalize_embeddings=True,
    )[0]

    similarities = evidence_embeddings @ query_embedding
    top_k = strategy["top_k"]
    top_indices = np.argsort(similarities)[::-1][:top_k]

    candidates = evidence_chunks.iloc[top_indices].copy()
    candidates["similarity"] = similarities[top_indices]

    candidates["context_keyword_score"] = candidates["text"].apply(
        lambda t: contextual_keyword_score(t, strategy["category_weights"])
    )

    max_score = candidates["context_keyword_score"].max()
    candidates["context_score"] = (
        candidates["context_keyword_score"] / max_score if max_score > 0 else 0.0
    )

    return candidates.reset_index(drop=True)


## 5. Evidence Ranking (named, traceable factors — not two fixed weights)

Combines semantic similarity, the severity-weighted context-keyword score,
and source authority into one ranking score. `RANKING_WEIGHTS` is explicit
and easy to justify/ablate in the paper, instead of a single unlabeled
`ALPHA`/`BETA` pair.

In [8]:
def classify_source(source):
    source = str(source).lower()
    # NOTE: check more specific substrings ("iaea_case") before the more
    # general ones ("iaea") -- "IAEA_Case_Istanbul" contains "iaea", so if
    # the generic check ran first it would misclassify every historical
    # case as the IAEA safety-standard category instead.
    if "iaea_case" in source or "historical" in source:
        return "historical_accident_case"
    elif "icrp" in source:
        return "radiation_protection_standard"
    elif "iaea" in source:
        return "radiation_safety_standard"
    elif "radiation_health" in source:
        return "health_effects_review"
    elif "who" in source:
        return "health_authority"
    elif "nrc" in source or "cfr" in source or "epa" in source:
        return "government_regulation"
    elif "ehs" in source or "safety_manual" in source or "sop" in source:
        return "institutional_sop"
    elif "secondary_review" in source:
        return "secondary_review"
    return "other"


SOURCE_WEIGHTS = {
    "radiation_protection_standard": 1.00,
    "radiation_safety_standard": 1.00,
    "health_authority": 1.00,
    "government_regulation": 0.95,   # binding national regulation
    "health_effects_review": 0.90,
    "historical_accident_case": 0.85,  # real IAEA-published case, but a single
                                        # incident rather than general guidance
    "institutional_sop": 0.85,
    "secondary_review": 0.55,  # uncertain journal legitimacy -- see prior discussion
    "other": 0.70,
}

# EXTENSION POINT: add "exposure_duration_match", "radiation_type_match",
# "location_match", "publication_recency" once that metadata exists in the
# corpus and in the situation profile, and give each its own named weight.
RANKING_WEIGHTS = {
    "similarity": 0.55,
    "context": 0.25,
    "source_authority": 0.20,
}


def rank_evidence(candidates, profile):
    ranked = candidates.copy()

    ranked["evidence_type"] = ranked["source"].apply(classify_source)
    ranked["source_weight"] = ranked["evidence_type"].map(SOURCE_WEIGHTS).fillna(0.70)

    ranked["ranking_score"] = (
        RANKING_WEIGHTS["similarity"] * ranked["similarity"]
        + RANKING_WEIGHTS["context"] * ranked["context_score"]
        + RANKING_WEIGHTS["source_authority"] * ranked["source_weight"]
    )

    ranked = ranked.sort_values("ranking_score", ascending=False).reset_index(drop=True)
    ranked["rank"] = np.arange(len(ranked)) + 1
    return ranked


## 6. Evidence Fusion (page-level dedup + conflict/consensus detection)

Two upgrades over the original fusion step:

1. **Page-level merge**: with `CHUNK_OVERLAP=200`, several overlapping chunks
   from the same `(source, page)` often make it into the top-K. Instead of
   citing near-duplicates as if they were independent evidence, keep only the
   single highest-ranked chunk per page.
2. **Conflict detection**: a lightweight keyword heuristic flags when the
   retrieved evidence contains both reassuring and cautionary language, so
   the LLM is explicitly told to surface disagreement rather than silently
   pick a side — matching the WHO-says-safe / SOP-says-evacuate scenario in
   your project notes, generalized to whatever sources you actually have.

In [9]:
REASSURING_TERMS = [
    "no significant risk", "not expected to cause harm", "within acceptable limits",
    "below the threshold", "considered safe", "acceptable level",
]

CONCERNING_TERMS = [
    "evacuate", "immediate action", "emergency", "exceeds the limit",
    "harmful", "seek medical attention", "stop exposure", "urgent",
]


def tag_stance(text):
    text_lower = text.lower()
    has_reassuring = any(term in text_lower for term in REASSURING_TERMS)
    has_concerning = any(term in text_lower for term in CONCERNING_TERMS)

    if has_reassuring and has_concerning:
        return "mixed"
    elif has_concerning:
        return "concerning"
    elif has_reassuring:
        return "reassuring"
    return "neutral"


def fuse_evidence(ranked, top_k=10):
    # Page-level dedup: keep only the top-ranked chunk per (source, page)
    deduped = (
        ranked.sort_values("rank")
        .drop_duplicates(subset=["source", "page"], keep="first")
        .reset_index(drop=True)
    )

    fused = deduped.head(top_k).copy()
    fused["support_strength"] = fused["ranking_score"] * fused["source_weight"]
    fused = fused.sort_values("support_strength", ascending=False).reset_index(drop=True)
    fused["fusion_rank"] = np.arange(len(fused)) + 1

    fused["stance"] = fused["text"].apply(tag_stance)

    stances_present = set(fused["stance"]) - {"neutral"}
    conflict_detected = {"reassuring", "concerning"}.issubset(stances_present) or "mixed" in stances_present

    return fused, conflict_detected


## 7. Lightweight claim extraction (stand-in for the NER/RE "NLP layer")

Rather than citing a whole 1200-character chunk, pull out the 1–3 sentences
per chunk most likely to be an actual claim (contains a domain keyword),
shrinking what's sent to the LLM and making citations point at something
closer to an atomic statement. This is a rule-based approximation of the
NER/relation-extraction step in your project notes — swap in spaCy/scispaCy
here if you want real entity-level extraction later; the rest of the
pipeline doesn't care how `claims` gets populated.

In [10]:
CLAIM_KEYWORDS = [
    "dose", "msv", "exposure", "limit", "recommend", "risk", "effect",
    "evacuate", "monitor", "guideline", "threshold", "protection", "safety",
]


def extract_claim_sentences(text, max_claims=3, min_len=25):
    # Simple sentence splitter — good enough for PDF-extracted prose.
    sentences = re.split(r'(?<=[.!?])\s+', text)

    scored = []
    for sentence in sentences:
        sentence = sentence.strip()
        if len(sentence) < min_len:
            continue
        lower = sentence.lower()
        score = sum(1 for kw in CLAIM_KEYWORDS if kw in lower)
        if score > 0:
            scored.append((score, sentence))

    scored.sort(key=lambda pair: pair[0], reverse=True)

    if not scored:
        # Nothing scored — fall back to the first couple of sentences
        # rather than dropping the evidence item entirely.
        return [s.strip() for s in sentences if len(s.strip()) >= min_len][:max_claims]

    return [sentence for _, sentence in scored[:max_claims]]


## 8. End-to-end: build the CARE++ evidence package for one situation

In [11]:
def build_evidence_package(row, evidence_chunks, evidence_embeddings, embedding_model, top_k=10):
    profile = build_situation_profile(row)

    candidates = retrieve_adaptive(profile, evidence_chunks, evidence_embeddings, embedding_model)
    ranked = rank_evidence(candidates, profile)
    fused, conflict_detected = fuse_evidence(ranked, top_k=top_k)

    evidence_items = []
    for _, r in fused.iterrows():
        evidence_items.append({
            "rank": int(r["fusion_rank"]),
            "source": str(r["source"]),
            "page": int(r["page"]),
            "evidence_type": str(r["evidence_type"]),
            "ranking_score": float(r["ranking_score"]),
            "support_strength": float(r["support_strength"]),
            "stance": str(r["stance"]),
            "claims": extract_claim_sentences(r["text"]),
            "full_text": str(r["text"]),  # kept for audit/traceability, not sent to the LLM prompt
        })

    return {
        "situation": profile,
        "conflict_detected": bool(conflict_detected),
        "evidence": evidence_items,
    }


In [12]:
# Worked example on the first test-set observation
example_row = context_df.iloc[0]

evidence_package = build_evidence_package(
    example_row, evidence_chunks, evidence_embeddings, embedding_model, top_k=10,
)

print("Severity tier:", evidence_package["situation"]["severity_tier"])
print("Conflict detected among retrieved evidence:", evidence_package["conflict_detected"])
print("\nEvidence (rank | source | page | stance | support_strength):")
for item in evidence_package["evidence"]:
    print(f"  {item['rank']:>2} | {item['source']:<24} | p.{item['page']:<4} | "
          f"{item['stance']:<10} | {item['support_strength']:.3f}")

print("\nExample claims extracted from the top item:")
for c in evidence_package["evidence"][0]["claims"]:
    print(" -", c)


Severity tier: high
Conflict detected among retrieved evidence: False

Evidence (rank | source | page | stance | support_strength):
   1 | IAEA                     | p.438  | concerning | 0.832
   2 | IAEA                     | p.34   | concerning | 0.811
   3 | IAEA                     | p.31   | neutral    | 0.791
   4 | IAEA                     | p.125  | neutral    | 0.789
   5 | ICRP_Publication_103     | p.4    | concerning | 0.787
   6 | IAEA                     | p.32   | neutral    | 0.764
   7 | IAEA                     | p.436  | neutral    | 0.757
   8 | ICRP_Publication_103     | p.16   | neutral    | 0.755
   9 | IAEA                     | p.42   | concerning | 0.740
  10 | EHS_Stanford_Safety_Manual | p.78   | neutral    | 0.629

Example claims extracted from the top item:
 - The protection of 
people from harmful effects of exposure to ionizing radiation, and the 
means for achieving this.
 - protection and safety
The protection of people against exposure to ionizing ra

## 9. Feed into the LLM prompt (compatible with your existing generation cells)

Same `situation` + `evidence` JSON shape your original `care_llm_input.json`
used, so your existing model-loading, generation, and safety-check cells
downstream don't need to change — only what's *inside* `evidence` is richer
now (claims instead of full chunks, plus a `conflict_detected` flag you can
surface in the prompt).

In [13]:
def build_llm_input(evidence_package):
    citation_instruction = """
EVIDENCE CITATION RULES:

You MUST cite evidence-derived claims using the exact source and page
supplied in the evidence records, in the form [SOURCE, p. PAGE].
Do NOT invent page numbers. Do NOT cite a page not supplied in the evidence.

Every factual statement about radiation protection, radiation health
effects, radiation risks, dose concepts, or safety guidance drawn from the
retrieved evidence MUST include a citation. The ML prediction itself does
NOT require a citation.

Clearly distinguish: (1) measured radiation data, (2) ML prediction,
(3) contextual information, (4) authoritative evidence.

CPM must NOT be converted into absorbed dose, equivalent dose, or effective
dose unless a valid detector calibration/conversion relationship is
explicitly provided.
"""

    if evidence_package["conflict_detected"]:
        citation_instruction += """
CONFLICT NOTICE: the retrieved evidence contains both cautionary and
reassuring language. Do NOT resolve this disagreement yourself. State
explicitly that the retrieved sources differ in tone/recommendation, cite
each side, and recommend deferring to a qualified radiation-protection
professional for a final determination.
"""

    task = """
Interpret the current radiation measurement using: (1) the measured CPM
value, (2) the machine-learning prediction, (3) prediction residual and
error, (4) temporal/contextual information, (5) retrieved authoritative
radiation evidence.

Produce a concise, scientifically cautious, evidence-grounded
interpretation that distinguishes observation from prediction and does not
infer that a stable trend means radiation exposure is safe.
"""

    # Evidence sent to the LLM uses extracted claims, not full chunk text,
    # to keep the prompt compact and citations closer to atomic statements.
    llm_evidence = []
    for item in evidence_package["evidence"]:
        llm_evidence.append({
            "rank": item["rank"],
            "source": item["source"],
            "page": item["page"],
            "evidence_type": item["evidence_type"],
            "support_strength": item["support_strength"],
            "stance": item["stance"],
            "text": " ".join(item["claims"]),
        })

    return {
        "situation": evidence_package["situation"],
        "conflict_detected": evidence_package["conflict_detected"],
        "task": task.strip(),
        "citation_instruction": citation_instruction.strip(),
        "evidence": llm_evidence,
    }


care_llm_input = build_llm_input(evidence_package)

os.makedirs("/content/care_prediction_models", exist_ok=True)
LLM_INPUT_PATH = "/content/care_prediction_models/care_llm_input.json"

with open(LLM_INPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(care_llm_input, f, indent=2, ensure_ascii=False)

print("Saved:", LLM_INPUT_PATH)
print("Evidence records:", len(care_llm_input["evidence"]))
print("Conflict detected:", care_llm_input["conflict_detected"])
# From here, your existing "CARE++ — LOAD HUGGING FACE LLM" and prompt-building
# cells can load this file exactly as before.


Saved: /content/care_prediction_models/care_llm_input.json
Evidence records: 10
Conflict detected: False


## 10. Batch mode (build packages for many/all test-set situations)

The worked example above does one row. For evaluation you'll want this
across the test set — this just loops `build_evidence_package` and reports
aggregate stats you can put in the paper (severity distribution, conflict
rate, average evidence-source diversity).

In [14]:
def build_evidence_packages_batch(context_df, evidence_chunks, evidence_embeddings, embedding_model,
                                    top_k=10, limit=None):
    rows = context_df if limit is None else context_df.head(limit)
    packages = []
    for _, row in rows.iterrows():
        packages.append(
            build_evidence_package(row, evidence_chunks, evidence_embeddings, embedding_model, top_k=top_k)
        )
    return packages


# Start small — this calls the embedding model once per row.
batch_packages = build_evidence_packages_batch(
    context_df, evidence_chunks, evidence_embeddings, embedding_model, top_k=10, limit=50,
)

severity_counts = pd.Series([p["situation"]["severity_tier"] for p in batch_packages]).value_counts()
conflict_rate = np.mean([p["conflict_detected"] for p in batch_packages])
avg_source_diversity = np.mean([
    pd.Series([e["source"] for e in p["evidence"]]).nunique() for p in batch_packages
])

print("Severity tier distribution (first", len(batch_packages), "rows):")
print(severity_counts)
print("\nConflict-detected rate:", round(conflict_rate, 3))
print("Average distinct sources per fused evidence set:", round(avg_source_diversity, 2))


Severity tier distribution (first 50 rows):
high        41
critical     9
Name: count, dtype: int64

Conflict-detected rate: 0.0
Average distinct sources per fused evidence set: 2.82


## 11. Load the LLM (Step 7 of the framework: "the LLM doesn't invent advice, it only explains retrieved evidence")

This is a fresh implementation rather than a copy of your original
`CARE___evidence_layer.ipynb` LLM cells — that notebook wasn't re-uploaded
alongside this one, so rather than guess at code I can't see anymore, this
rebuilds the step from scratch against the same `care_llm_input` JSON shape
Section 9 produces. If your original notebook's prompt wording differs and
you want this to match it exactly, upload it and I'll reconcile the two.

Swap `MODEL_ID` for whatever you were already using (e.g. a quantized
Qwen2.5 checkpoint) — nothing else here depends on which model it is.

In [15]:
#!pip install -U "transformers==4.57.3" "accelerate>=1.2.0" "bitsandbytes>=0.50.0"

In [16]:
import torch

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig
)

MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

llm_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quant_config,
    device_map="auto",
)

print("Loaded:", MODEL_ID)

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

...

model-00002-of-00002.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/3.97G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Loaded: Qwen/Qwen2.5-3B-Instruct


## 12. Build the grounded prompt from `care_llm_input`

In [17]:
def format_evidence_for_prompt(evidence_list):
    lines = []
    for item in evidence_list:
        lines.append(
            f"[{item['source']}, p. {item['page']}] "
            f"(type: {item['evidence_type']}, stance: {item['stance']}, "
            f"support strength: {item['support_strength']:.2f})\n"
            f"{item['text']}"
        )
    return "\n\n".join(lines)


def build_grounded_prompt(care_llm_input):
    situation = care_llm_input["situation"]

    situation_text = f"""
Measured radiation: {situation['current_cpm']:.2f} CPM
ML-predicted next value: {situation['predicted_cpm']:.2f} CPM
Prediction residual: {situation['prediction_residual']:.2f} CPM (direction: {situation['prediction_error_direction']})
Relative prediction error: {situation['relative_prediction_error']:.3f}
Trend: {situation['trend']}
Time: {situation['time_period']} ({situation['day_name']})
Severity tier (retrieval-routing only, not a safety verdict): {situation['severity_tier']}
""".strip()

    evidence_text = format_evidence_for_prompt(care_llm_input["evidence"])

    prompt = f"""You are an assistant that explains radiation monitoring data using ONLY the evidence provided below. You do not have independent knowledge of radiation safety limits beyond what is cited here.

TASK:
{care_llm_input['task']}

{care_llm_input['citation_instruction']}

CURRENT SITUATION:
{situation_text}

RETRIEVED EVIDENCE:
{evidence_text}

Write a concise, cautious, evidence-grounded interpretation now. Every claim about radiation protection, health effects, or safety guidance must carry a [SOURCE, p. PAGE] citation copied exactly from the evidence above.
"""
    return prompt.strip()


prompt = build_grounded_prompt(care_llm_input)
print(prompt[:2000])
print("\n... [prompt continues] ...")


You are an assistant that explains radiation monitoring data using ONLY the evidence provided below. You do not have independent knowledge of radiation safety limits beyond what is cited here.

TASK:
Interpret the current radiation measurement using: (1) the measured CPM
value, (2) the machine-learning prediction, (3) prediction residual and
error, (4) temporal/contextual information, (5) retrieved authoritative
radiation evidence.

Produce a concise, scientifically cautious, evidence-grounded
interpretation that distinguishes observation from prediction and does not
infer that a stable trend means radiation exposure is safe.

EVIDENCE CITATION RULES:

You MUST cite evidence-derived claims using the exact source and page
supplied in the evidence records, in the form [SOURCE, p. PAGE].
Do NOT invent page numbers. Do NOT cite a page not supplied in the evidence.

Every factual statement about radiation protection, radiation health
effects, radiation risks, dose concepts, or safety guidan

## 13. Generate the explanation

In [18]:
def generate_explanation(prompt, max_new_tokens=400):
    messages = [{"role": "user", "content": prompt}]
    input_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(input_text, return_tensors="pt").to(llm_model.device)

    with torch.no_grad():
        output_ids = llm_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.3,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )

    generated = output_ids[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()


care_response = generate_explanation(prompt)
print(care_response)


### Interpretation of Current Radiation Measurement

#### Measured Radiation Data
- **Current Radiation Measurement:** 10.00 CPM
  - [IAEA, p. 438]

#### ML Prediction
- **Machine Learning Prediction:** 13.69 CPM
  - [IAEA, p. 438]
- **Prediction Residual:** -3.69 CPM (overprediction)
- **Relative Prediction Error:** 0.369
  - [IAEA, p. 438]

#### Contextual Information
- **Time:** Night (Monday)
  - [IAE, p. 438]
- **Severity Tier:** High
  - [IAE, p. 438]
- **Temporal Context:** Nighttime radiation levels often increase due to natural background radiation and human activities.
  - [IAE, p. 438]

#### Authoritative Evidence
- **Radiation Protection Standards:**
  - [IAE, p. 32]: These Standards consider the findings of the UNSCEAR and ICRP, maintaining the Commission's three fundamental principles of radiation protection: justification, optimization, and application of dose limits.
  - [IAE, p. 436]: Optimization involves ensuring that individual doses, the number of exposed individua

## 14. Citation and safety validation

Two independent checks on the generated text, both rule-based (not another
LLM call) so they're cheap to run on every output:

1. **Citation grounding** — every `[SOURCE, p. PAGE]` citation the model
   wrote must match a `(source, page)` pair that was actually retrieved.
   A citation to a page never in the evidence list means the model
   fabricated a page number, which the paper explicitly prohibits
   (Section IX).
2. **Unsupported dose-conversion / safety-verdict language** — flags
   phrases that would convert CPM into dose, or assert a safety
   conclusion, without the evidence explicitly licensing it. This is a
   heuristic safety net, not a guarantee — always spot-check output.

In [19]:
import re

def validate_citations(response_text, evidence_list):
    cited_pairs = set(re.findall(r"\[([^,\]]+),\s*p\.\s*(\d+)\]", response_text))
    retrieved_pairs = {(item["source"], str(item["page"])) for item in evidence_list}

    fabricated = cited_pairs - retrieved_pairs
    uncited_evidence = retrieved_pairs - cited_pairs

    return {
        "cited_pairs": cited_pairs,
        "fabricated_citations": fabricated,
        "unused_evidence": uncited_evidence,
        "all_citations_grounded": len(fabricated) == 0,
    }


UNSUPPORTED_PHRASES = [
    "definitely safe", "guaranteed safe", "no risk at all",
    "converts to", "equivalent to a dose of", "absorbed dose of",
    "you will experience", "this will cause",
]


def check_unsupported_claims(response_text):
    text_lower = response_text.lower()
    flagged = [p for p in UNSUPPORTED_PHRASES if p in text_lower]
    return {"flagged_phrases": flagged, "clean": len(flagged) == 0}


citation_check = validate_citations(care_response, care_llm_input["evidence"])
safety_check = check_unsupported_claims(care_response)

print("Citations found in response:", citation_check["cited_pairs"])
print("Fabricated citations (should be empty):", citation_check["fabricated_citations"])
print("All citations grounded:", citation_check["all_citations_grounded"])
print()
print("Unsupported-claim phrases flagged (should be empty):", safety_check["flagged_phrases"])
print("Passes safety heuristic:", safety_check["clean"])

if not citation_check["all_citations_grounded"] or not safety_check["clean"]:
    print("\n>>> This output needs a second look before use — do not treat it as final. <<<")


Citations found in response: {('IAE', '436'), ('IAEA', '438'), ('IAE', '438'), ('ICRP_Publication_103', '16'), ('IAE', '32')}
Fabricated citations (should be empty): {('IAE', '32'), ('IAE', '438'), ('IAE', '436')}
All citations grounded: False

Unsupported-claim phrases flagged (should be empty): []
Passes safety heuristic: True

>>> This output needs a second look before use — do not treat it as final. <<<


## 15. Batch mode: generate + validate for many situations

Loops Sections 12–14 over `batch_packages` from Section 10, so you can
report an aggregate citation-grounding rate and safety-flag rate across the
test set for the paper, instead of a single anecdotal example.

In [20]:
def run_care_pipeline(package):
    llm_input = build_llm_input(package)
    prompt = build_grounded_prompt(llm_input)
    response = generate_explanation(prompt)
    citation_result = validate_citations(response, llm_input["evidence"])
    safety_result = check_unsupported_claims(response)
    return {
        "llm_input": llm_input,
        "response": response,
        "citation_check": citation_result,
        "safety_check": safety_result,
    }


# Start small — each call is a full LLM generation.
batch_results = [run_care_pipeline(pkg) for pkg in batch_packages[:10]]

grounded_rate = sum(r["citation_check"]["all_citations_grounded"] for r in batch_results) / len(batch_results)
clean_rate = sum(r["safety_check"]["clean"] for r in batch_results) / len(batch_results)

print(f"Citation-grounding rate: {grounded_rate:.1%} ({len(batch_results)} samples)")
print(f"Safety-heuristic pass rate: {clean_rate:.1%} ({len(batch_results)} samples)")


Citation-grounding rate: 100.0% (10 samples)
Safety-heuristic pass rate: 100.0% (10 samples)


In [21]:
# CARE++ — EVIDENCE LAYER EVALUATION METRICS

import math
import numpy as np
import pandas as pd

def wilson_ci(successes, n, z=1.96):
    """95% Wilson score interval -- more honest than a raw percentage for
    small n, and standard practice for reporting a proportion in a paper."""
    if n == 0:
        return (0.0, 0.0)
    phat = successes / n
    denom = 1 + z**2 / n
    center = (phat + z**2 / (2 * n)) / denom
    margin = (z * math.sqrt(phat * (1 - phat) / n + z**2 / (4 * n**2))) / denom
    return (max(0.0, center - margin), min(1.0, center + margin))


# Evaluate on a larger, more defensible sample than the original 10
N_EVAL_SAMPLES = min(50, len(batch_packages))
print(f"Evaluating on {N_EVAL_SAMPLES} samples (each involves a full LLM generation call)...\n")

eval_results = [run_care_pipeline(pkg) for pkg in batch_packages[:N_EVAL_SAMPLES]]

# Retrieval / ranking / fusion metrics
severity_counts = pd.Series([p["situation"]["severity_tier"] for p in batch_packages[:N_EVAL_SAMPLES]]).value_counts()
conflict_rate = np.mean([p["conflict_detected"] for p in batch_packages[:N_EVAL_SAMPLES]])
source_diversity = [
    pd.Series([e["source"] for e in p["evidence"]]).nunique()
    for p in batch_packages[:N_EVAL_SAMPLES]
]
support_strengths = [e["support_strength"] for p in batch_packages[:N_EVAL_SAMPLES] for e in p["evidence"]]
claims_per_item = [len(e["claims"]) for p in batch_packages[:N_EVAL_SAMPLES] for e in p["evidence"]]
stance_counts = pd.Series([e["stance"] for p in batch_packages[:N_EVAL_SAMPLES] for e in p["evidence"]]).value_counts()

# LLM generation / citation grounding metrics
n_grounded = sum(r["citation_check"]["all_citations_grounded"] for r in eval_results)
n_safe = sum(r["safety_check"]["clean"] for r in eval_results)

citations_per_response = [len(r["citation_check"]["cited_pairs"]) for r in eval_results]
total_evidence_offered = [len(r["llm_input"]["evidence"]) for r in eval_results]
total_evidence_cited = [
    len(r["citation_check"]["cited_pairs"] & {(e["source"], str(e["page"])) for e in r["llm_input"]["evidence"]})
    for r in eval_results
]
utilization_rates = [
    cited / offered if offered > 0 else 0.0
    for cited, offered in zip(total_evidence_cited, total_evidence_offered)
]

grounded_rate = n_grounded / N_EVAL_SAMPLES
safe_rate = n_safe / N_EVAL_SAMPLES
grounded_ci = wilson_ci(n_grounded, N_EVAL_SAMPLES)
safe_ci = wilson_ci(n_safe, N_EVAL_SAMPLES)

# Print a clean, paper-ready summary
print("=" * 70)
print(f"CARE++ EVIDENCE LAYER EVALUATION (n = {N_EVAL_SAMPLES})")
print("=" * 70)

print("\n--- Situation characteristics ---")
print("Severity tier distribution:")
print((severity_counts / N_EVAL_SAMPLES * 100).round(1).astype(str) + "%")

print("\n--- Retrieval / ranking / fusion ---")
print(f"Conflict-detected rate:              {conflict_rate:.1%}")
print(f"Avg. distinct sources per situation:  {np.mean(source_diversity):.2f} (std {np.std(source_diversity):.2f})")
print(f"Support strength -- mean/median/std:  {np.mean(support_strengths):.3f} / "
      f"{np.median(support_strengths):.3f} / {np.std(support_strengths):.3f}")
print(f"Claims extracted per evidence item:   {np.mean(claims_per_item):.2f} (std {np.std(claims_per_item):.2f})")
print("Evidence stance distribution:")
print((stance_counts / stance_counts.sum() * 100).round(1).astype(str) + "%")

print("\n--- LLM generation / citation grounding ---")
print(f"Citation-grounding rate:  {grounded_rate:.1%}  (95% CI: {grounded_ci[0]:.1%}-{grounded_ci[1]:.1%})")
print(f"Safety-heuristic pass rate: {safe_rate:.1%}  (95% CI: {safe_ci[0]:.1%}-{safe_ci[1]:.1%})")
print(f"Avg. citations per response: {np.mean(citations_per_response):.2f} (std {np.std(citations_per_response):.2f})")
print(f"Evidence utilization rate (cited / offered): {np.mean(utilization_rates):.1%} "
      f"(std {np.std(utilization_rates):.1%})")

print("\n--- Caveat for the paper ---")
print(f"n={N_EVAL_SAMPLES} is still modest for a headline number -- consider running on the")
print("full test set (or at least n=200+) before citing these as final results.")
print("Report the confidence interval alongside the point estimate, not just the percentage.")

# Table form, ready to paste into a LaTeX/Word table
summary_table = pd.DataFrame([
    {"Metric": "Conflict-detected rate", "Value": f"{conflict_rate:.1%}"},
    {"Metric": "Avg. distinct sources / situation", "Value": f"{np.mean(source_diversity):.2f}"},
    {"Metric": "Avg. claims / evidence item", "Value": f"{np.mean(claims_per_item):.2f}"},
    {"Metric": "Citation-grounding rate", "Value": f"{grounded_rate:.1%} (95% CI {grounded_ci[0]:.1%}-{grounded_ci[1]:.1%})"},
    {"Metric": "Safety-heuristic pass rate", "Value": f"{safe_rate:.1%} (95% CI {safe_ci[0]:.1%}-{safe_ci[1]:.1%})"},
    {"Metric": "Evidence utilization rate", "Value": f"{np.mean(utilization_rates):.1%}"},
])
print("\n" + summary_table.to_string(index=False))

Evaluating on 50 samples (each involves a full LLM generation call)...

CARE++ EVIDENCE LAYER EVALUATION (n = 50)

--- Situation characteristics ---
Severity tier distribution:
high        82.0%
critical    18.0%
Name: count, dtype: object

--- Retrieval / ranking / fusion ---
Conflict-detected rate:              0.0%
Avg. distinct sources per situation:  2.82 (std 0.38)
Support strength -- mean/median/std:  0.769 / 0.784 / 0.049
Claims extracted per evidence item:   2.90 (std 0.30)
Evidence stance distribution:
neutral       61.4%
concerning    38.6%
Name: count, dtype: object

--- LLM generation / citation grounding ---
Citation-grounding rate:  100.0%  (95% CI: 92.9%-100.0%)
Safety-heuristic pass rate: 100.0%  (95% CI: 92.9%-100.0%)
Avg. citations per response: 2.72 (std 1.97)
Evidence utilization rate (cited / offered): 27.2% (std 19.7%)

--- Caveat for the paper ---
n=50 is still modest for a headline number -- consider running on the
full test set (or at least n=200+) before citi